# SAM-Audio - Colab lineare

Notebook ottimizzato per separare una sorgente audio con SAM-Audio usando un prompt testuale.

Flusso consigliato: esegui le celle dall'alto verso il basso. Le impostazioni avanzate piu' fragili (`predict_spans` e `reranking_candidates`) sono disattivate per tenere basso l'uso di VRAM e ridurre gli errori su GPU Colab T4.

**Requisiti:**
- Runtime Colab con GPU (`Runtime > Change runtime type > T4 GPU` o superiore)
- Token Hugging Face con accesso approvato al checkpoint SAM-Audio scelto

## 1. Setup

In [ ]:
#@title 1. Setup SAM-Audio (esegui una volta)
import os
import pathlib
import re
import subprocess
import sys

REPO_URL = "https://github.com/facebookresearch/sam-audio.git"
REPO_DIR = pathlib.Path("/content/sam-audio")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo gia' presente: {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Cartella di lavoro: {pathlib.Path.cwd()}")

pip_cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    ".",
    "numpy>=2.0,<2.1",
    "numba>=0.60",
    "huggingface_hub>=0.34,<1.0",
    "transformers>=4.54,<5",
    "protobuf",
    "tensorboard",
]
subprocess.run(pip_cmd, check=True)

# Compatibilita' con versioni recenti di huggingface_hub.
# SAM-Audio upstream richiede ancora argomenti che alcune versioni hub non passano piu'.
base_file = REPO_DIR / "sam_audio" / "model" / "base.py"
source = base_file.read_text(encoding="utf-8")
source = source.replace("proxies: Optional[Dict],", "proxies: Optional[Dict] = None,")
source = source.replace("resume_download: bool,", "resume_download: bool = False,")
source = re.sub(r"\n\s+proxies=proxies,", "", source)
source = re.sub(r"\n\s+resume_download=resume_download,", "", source)

if re.search(r"\bproxies=proxies\b|\bresume_download=resume_download\b", source):
    raise RuntimeError("Patch Hugging Face non applicata: controlla sam_audio/model/base.py")

base_file.write_text(source, encoding="utf-8")
print("Setup completato. Continua con il login Hugging Face.")

## 2. Login Hugging Face

Se usi Colab Secrets, salva il token come `HF_TOKEN`. Altrimenti la cella aprira' il login interattivo.

In [ ]:
#@title 2. Login Hugging Face
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    login(token=token)
    print("Login completato usando Colab Secret HF_TOKEN.")
else:
    print("Inserisci un token Hugging Face con accesso al modello scelto.")
    login()

## 3. Impostazioni

In [ ]:
#@title 3. Impostazioni principali
MODEL_ID = "facebook/sam-audio-base" #@param ["facebook/sam-audio-small", "facebook/sam-audio-base", "facebook/sam-audio-large"]
DESCRIPTION = "man speaking" #@param {type:"string"}
CHUNK_SECONDS = 30 #@param {type:"slider", min:10, max:90, step:10}
AUDIO_SOURCE = "upload" #@param ["upload", "google_drive"]
DRIVE_AUDIO_PATH = "" #@param {type:"string"}
DOWNLOAD_RESIDUAL = True #@param {type:"boolean"}

DESCRIPTION = DESCRIPTION.strip().lower()
CHUNK_SECONDS = int(CHUNK_SECONDS)

if not DESCRIPTION:
    raise ValueError("DESCRIPTION non puo' essere vuota. Esempi: 'man speaking', 'drums', 'guitar'.")
if AUDIO_SOURCE == "google_drive" and not DRIVE_AUDIO_PATH.strip():
    raise ValueError("Imposta DRIVE_AUDIO_PATH oppure cambia AUDIO_SOURCE in 'upload'.")
if "large" in MODEL_ID and CHUNK_SECONDS > 30:
    print("Nota: il modello large usa piu' VRAM. Se compare OOM, porta CHUNK_SECONDS a 10 o 20.")

print(f"Modello: {MODEL_ID}")
print(f"Prompt: {DESCRIPTION!r}")
print(f"Chunk: {CHUNK_SECONDS}s")
print(f"Sorgente audio: {AUDIO_SOURCE}")

## 4. Caricamento modello

In [ ]:
#@title 4. Carica il modello in modalita' leggera
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import types
import torch
import torchaudio
from sam_audio import SAMAudio, SAMAudioProcessor

if not torch.cuda.is_available():
    raise RuntimeError("GPU non disponibile. In Colab abilita: Runtime > Change runtime type > T4 GPU.")

DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True

if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

processor = SAMAudioProcessor.from_pretrained(MODEL_ID)
model = SAMAudio.from_pretrained(MODEL_ID).eval().to(DEVICE)

# Riduce la memoria: modello in fp16, codec audio in fp32 per evitare errori conv1d.
model = model.half()
model.audio_codec = model.audio_codec.float()

# Modalita' lineare: niente reranking/spans, quindi questi moduli non servono.
for module_name in ("visual_ranker", "text_ranker", "span_predictor"):
    if hasattr(model, module_name):
        delattr(model, module_name)

# Codec in fp32, resto del modello in fp16.
def _get_audio_features_patched(self, audios: torch.Tensor):
    audio_features = self.audio_codec(audios.float()).transpose(1, 2).half()
    return torch.cat([audio_features, audio_features], dim=2)

model._get_audio_features = types.MethodType(_get_audio_features_patched, model)

# Decode un elemento alla volta: piu' lento, ma molto piu' stabile su T4.
_original_decode = model.audio_codec.decode

def _decode_patched(encoded_frames):
    decoded = []
    for index in range(encoded_frames.size(0)):
        decoded.append(_original_decode(encoded_frames[index:index + 1].float()))
        torch.cuda.empty_cache()
    return torch.cat(decoded, dim=0)

model.audio_codec.decode = _decode_patched

gc.collect()
torch.cuda.empty_cache()

SAMPLE_RATE = processor.audio_sampling_rate
print(f"Modello caricato: {MODEL_ID}")
print(f"Sample rate output: {SAMPLE_RATE} Hz")
print(f"VRAM usata: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 5. Audio input

In [ ]:
#@title 5. Carica audio e crea chunk
from pathlib import Path
from IPython.display import Audio, display
import math
import shutil
from google.colab import files

INPUT_DIR = Path("/content/sam_audio_input")
CHUNK_DIR = Path("/content/sam_audio_chunks")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

if CHUNK_DIR.exists():
    shutil.rmtree(CHUNK_DIR)
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

if AUDIO_SOURCE == "upload":
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Nessun file caricato.")
    uploaded_name = next(iter(uploaded.keys()))
    source_path = Path(uploaded_name)
    AUDIO_PATH = INPUT_DIR / source_path.name
    if AUDIO_PATH.exists():
        AUDIO_PATH.unlink()
    shutil.move(str(source_path), str(AUDIO_PATH))
else:
    from google.colab import drive
    drive.mount("/content/drive")
    AUDIO_PATH = Path(DRIVE_AUDIO_PATH.strip()).expanduser()
    if not AUDIO_PATH.exists():
        raise FileNotFoundError(f"File non trovato: {AUDIO_PATH}")

waveform, source_sr = torchaudio.load(str(AUDIO_PATH))
if waveform.numel() == 0:
    raise ValueError("Il file audio sembra vuoto.")

duration = waveform.shape[-1] / source_sr
print(f"File: {AUDIO_PATH}")
print(f"Durata: {duration:.1f}s | Canali: {waveform.shape[0]} | Sample rate sorgente: {source_sr} Hz")
display(Audio(str(AUDIO_PATH)))

chunk_samples = max(1, int(CHUNK_SECONDS * source_sr))
total_samples = waveform.shape[-1]
AUDIO_CHUNKS = []

if total_samples > chunk_samples:
    num_chunks = math.ceil(total_samples / chunk_samples)
    print(f"Audio lungo: creo {num_chunks} chunk da circa {CHUNK_SECONDS}s.")
    for index in range(num_chunks):
        start = index * chunk_samples
        end = min(start + chunk_samples, total_samples)
        chunk_path = CHUNK_DIR / f"chunk_{index + 1:03d}.wav"
        torchaudio.save(str(chunk_path), waveform[:, start:end], source_sr)
        AUDIO_CHUNKS.append(str(chunk_path))
        print(f"  {chunk_path.name}: {start / source_sr:.1f}s - {end / source_sr:.1f}s")
else:
    AUDIO_CHUNKS = [str(AUDIO_PATH)]
    print("Audio breve: nessun chunk necessario.")

print(f"Chunk pronti: {len(AUDIO_CHUNKS)}")

## 6. Separazione

In [ ]:
#@title 6. Separa audio e salva risultati
from pathlib import Path
import gc
import re
import zipfile
import torch
import torchaudio

OUTPUT_DIR = Path("/content/sam_audio_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _slugify(text):
    slug = re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")
    return slug or "target"


def _as_audio_tensor(value):
    if isinstance(value, (list, tuple)):
        value = value[0]
    value = value.detach().cpu()
    while value.dim() > 2 and value.shape[0] == 1:
        value = value.squeeze(0)
    if value.dim() == 1:
        value = value.unsqueeze(0)
    if value.dim() != 2:
        raise ValueError(f"Forma audio inattesa: {tuple(value.shape)}")
    return value.float().contiguous()


def _concat_audio(paths, output_path):
    waves = []
    expected_sr = None
    for path in paths:
        wave, sr = torchaudio.load(str(path))
        if expected_sr is None:
            expected_sr = sr
        elif sr != expected_sr:
            raise ValueError("Sample rate diverso tra i chunk di output.")
        if wave.dim() == 1:
            wave = wave.unsqueeze(0)
        waves.append(wave)

    channels = max(wave.shape[0] for wave in waves)
    normalized = []
    for wave in waves:
        if wave.shape[0] == channels:
            normalized.append(wave)
        elif wave.shape[0] == 1:
            normalized.append(wave.repeat(channels, 1))
        else:
            raise ValueError("Numero di canali incompatibile tra i chunk.")

    combined = torch.cat(normalized, dim=-1)
    torchaudio.save(str(output_path), combined, expected_sr)
    return output_path


def separate_audio(description, prefix=None):
    description = description.strip().lower()
    if not description:
        raise ValueError("La descrizione non puo' essere vuota.")

    prefix = _slugify(prefix or description)
    target_paths = []
    residual_paths = []

    for index, chunk_path in enumerate(AUDIO_CHUNKS, start=1):
        print(f"[{index}/{len(AUDIO_CHUNKS)}] {Path(chunk_path).name} -> {description!r}")
        batch = processor(
            audios=[chunk_path],
            descriptions=[description],
        ).to(DEVICE)

        with torch.inference_mode():
            result = model.separate(
                batch,
                predict_spans=False,
                reranking_candidates=1,
            )

        target = _as_audio_tensor(result.target)
        residual = _as_audio_tensor(result.residual)

        target_path = OUTPUT_DIR / f"{prefix}_target_chunk_{index:03d}.wav"
        residual_path = OUTPUT_DIR / f"{prefix}_residual_chunk_{index:03d}.wav"
        torchaudio.save(str(target_path), target, SAMPLE_RATE)
        torchaudio.save(str(residual_path), residual, SAMPLE_RATE)
        target_paths.append(target_path)
        residual_paths.append(residual_path)

        del batch, result, target, residual
        gc.collect()
        torch.cuda.empty_cache()

    target_file = OUTPUT_DIR / f"{prefix}_target.wav"
    residual_file = OUTPUT_DIR / f"{prefix}_residual.wav"
    _concat_audio(target_paths, target_file)
    _concat_audio(residual_paths, residual_file)

    zip_file = OUTPUT_DIR / f"{prefix}_sam_audio_results.zip"
    with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(target_file, arcname=target_file.name)
        archive.write(residual_file, arcname=residual_file.name)
        for path in target_paths + residual_paths:
            archive.write(path, arcname=path.name)

    print("Separazione completata.")
    print(f"Target: {target_file}")
    print(f"Residuo: {residual_file}")
    print(f"Zip: {zip_file}")
    return {
        "description": description,
        "target": target_file,
        "residual": residual_file,
        "zip": zip_file,
        "target_chunks": target_paths,
        "residual_chunks": residual_paths,
    }


RESULT = separate_audio(DESCRIPTION)

## 7. Ascolto e download

In [ ]:
#@title 7. Ascolta e scarica
from IPython.display import Audio, display
from google.colab import files

print(f"Descrizione: {RESULT['description']!r}")
print("Target isolato")
display(Audio(str(RESULT["target"])))

print("Residuo")
display(Audio(str(RESULT["residual"])))

files.download(str(RESULT["target"]))
if DOWNLOAD_RESIDUAL:
    files.download(str(RESULT["residual"]))
files.download(str(RESULT["zip"]))

## 8. Opzionale: piu' prompt

Lascia il campo vuoto se non serve. Se inserisci piu' prompt separati da virgole, il notebook riusa lo stesso audio e genera un set di output per ciascuno.

In [ ]:
#@title 8. Separazioni multiple opzionali
MULTI_DESCRIPTIONS = "" #@param {type:"string"}

if MULTI_DESCRIPTIONS.strip():
    MULTI_RESULTS = []
    prompts = [item.strip().lower() for item in MULTI_DESCRIPTIONS.split(",") if item.strip()]
    for prompt in prompts:
        MULTI_RESULTS.append(separate_audio(prompt))
    print(f"Completate {len(MULTI_RESULTS)} separazioni multiple.")
else:
    print("Nessun prompt multiplo impostato: cella saltata.")